## Setup

In [4]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Connect

In [16]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.reload_config()
ble.connect()

2026-01-23 12:20:15,014 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:d1:79:ab:49
2026-01-23 12:20:15,015 | INFO     |: Scanning for device with address: c0:42:d1:79:ab:49, service UUID: 4843fd62-068c-4a37-af34-e724c6a05681
2026-01-23 12:20:25,235 | INFO     |: Found 360 total devices
2026-01-23 12:20:25,241 | INFO     |: Found matching device: C0:42:D1:79:AB:49 (name: Artemis BLE)
2026-01-23 12:20:26,973 | INFO     |: Connected to c0:42:d1:79:ab:49


## Echo

In [30]:
# Send the message
ble.send_command(CMD.ECHO, "Hello World!")

# Check that the board received it and returns it
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

Recieved from board: Hello World!


## Send Three Floats

In [27]:
ble.send_command(CMD.SEND_THREE_FLOATS, "3.14|5.25|7.432")

## Get Time Millis

In [19]:
# Send command
ble.send_command(CMD.GET_TIME_MILLIS, "")

# Check board message
s = ble.receive_string(ble.uuid['RX_STRING'])
print("Recieved from board: " + s)

Recieved from board: T:105686


## Notification Handler

In [35]:
def notification_callback(uuid, charac_bytearray):
    msg = ble.bytearray_to_string(charac_bytearray)
    if msg.startswith("T:"):
        t_ms = int(msg.split(":")[1])
        print("Time (ms):", t_ms)
    else:
        print("Unexpected message:", msg)

duration_s = 0.3  # set collection time

# Set up callback
ble.start_notify(ble.uuid['RX_STRING'], notification_callback)

t0 = time.time()
while (time.time() - t0) < duration_s:
    time.sleep(0.01)   # small sleep so you don't busy-wait

ble.stop_notify(ble.uuid['RX_STRING'])
print("End time notify")




Time (ms): 194043
Time (ms): 194043
Time (ms): 194043
Time (ms): 194044
Time (ms): 194101
Time (ms): 194162
Time (ms): 194162
Time (ms): 194283
Time (ms): 194283
Time (ms): 194283
Time (ms): 194338
Time (ms): 194345
Time (ms): 194401
Time (ms): 194401
Time (ms): 194401
Time (ms): 194402
End time notify


## Time Array

In [18]:
# Time array
time_values = []
receiving_msg = 0

def notification_callback(uuid, charac_bytearray):
    global receiving_msg
    msg = ble.bytearray_to_string(charac_bytearray).strip()
    print("Msg received:", repr(msg))
    #TODO split by commas and check for end

    parts = [p.strip() for p in msg.split(",") if p.strip()]

    if parts and parts[-1].lower() == "end":
        parts = parts[:-1]          # drop the "end"
        time_values.extend(parts)   # or convert to int below
        print("End of values")
        receiving_msg = 0
        return

    # normal chunk
    time_values.extend(parts)

    # if msg == "end":
    #     print("Finishing collection")
    #     receiving_msg = 0
    #     return

    # parts = msg.split(":")
    # if len(parts) == 2 and parts[0].isdigit():
    #     idx = int(parts[0])
    #     t_ms = int(parts[1])
    #     print(idx, t_ms)
    #     time_values.append(t_ms)
    # else:
    #     print("Unexpected message:", msg)

# TODO ack? or recieve chunks

# Set up callback
ble.start_notify(ble.uuid['RX_STRING'], notification_callback)

# Send command
ble.send_command(CMD.SEND_TIME_DATA, "")

# Receive data
receiving_msg = 1

timeout_s = 10.0
t0 = time.time()

# TODO figure out how to fix this so it just finishes when it hits the end
# TODO also figure out why this only works once
while receiving_msg == 1 and (time.time() - t0) < timeout_s:
    time.sleep(0.01)

if receiving_msg == 1:
    print("Timeout reached, stopping notify.")
    receiving_msg = 0
    
ble.stop_notify(ble.uuid['RX_STRING'])
print("End time notify")

print(time_values)


Timeout reached, stopping notify.
End time notify
[]
